In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/resnet_classifier

In [ ]:
!pwd

In [ ]:
!ls

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

In [ ]:
# Hyperparameters
RANDOM_SEED = 7
LEARNING_RATE = 0.01

# Architecture
NUM_FEATURES = 128 * 128
NUM_CLASSES = 2
BATCH_SIZE = 480
GRAYSCALE = False

Dataset Preparation

Change Hair-Color column to involve the three hair color types.

In [ ]:
path_1 = r"/content/drive/MyDrive/resnet_classifier/smile_0"
path_2 = r"/content/drive/MyDrive/resnet_classifier/smile_1"

In [ ]:
list_1 = []
list_2 = []

In [ ]:
for path in os.listdir(path_1):
    if os.path.isfile(os.path.join(path_1, path)):
        list_1.append(path)

In [ ]:
for path in os.listdir(path_2):
    if os.path.isfile(os.path.join(path_2, path)):
        list_2.append(path)

In [ ]:
label_1 = []
label_2 = []

In [ ]:
for i in range(len(list_1)):
    label_1.append(0)

In [ ]:
for i in range(len(list_2)):
    label_2.append(1)

In [ ]:
split_1 = []
split_2 = []

In [ ]:
for i in range(int(3 * len(list_1) / 5)):
    split_1.append(0)
    split_2.append(0)

In [ ]:
for i in range(int(len(list_1) / 5)):
    split_1.append(1)
    split_2.append(1)

In [ ]:
for i in range(int(len(list_1) / 5)):
    split_1.append(2)
    split_2.append(2)

In [ ]:
df_smile_0 = pd.DataFrame(list_1, columns=["Filename"])
df_smile_0 = df_smile_0.set_index("Filename")
df_smile_0["Smiling"] = label_1
df_smile_0["Partition"] = split_1
print(df_smile_0)
print(df_smile_0.iloc[5])

In [ ]:
df_smile_1 = pd.DataFrame(list_2, columns=["Filename"])
df_smile_1 = df_smile_1.set_index("Filename")
df_smile_1["Smiling"] = label_2
df_smile_1["Partition"] = split_2
print(df_smile_1)
print(df_smile_1.iloc[5])

In [ ]:
dataframes_smile_initial = [df_smile_0, df_smile_1]
dataframes_smile = pd.concat(dataframes_smile_initial)

In [ ]:
print(dataframes_smile)

In [ ]:
dataframes_smile.loc[dataframes_smile["Partition"] == 0].to_csv(
    "celeba-smile-train.csv"
)
dataframes_smile.loc[dataframes_smile["Partition"] == 1].to_csv(
    "celeba-smile-valid.csv"
)
dataframes_smile.loc[dataframes_smile["Partition"] == 2].to_csv("celeba-smile-test.csv")

In [ ]:
img = Image.open("smile_1/991.jpg")
print(np.asarray(img, dtype=np.uint8).shape)
plt.imshow(img);

DataLoader

In [ ]:
class CelebaDataset(Dataset):
    """Custom Dataset for loading CelebA face images"""

    def __init__(self, csv_path, img_dir, transform=None):

        df = pd.read_csv(csv_path, index_col=0)
        self.img_dir = img_dir
        self.csv_path = csv_path
        self.img_names = df.index.values
        self.y = df["Smiling"].values
        self.transform = transform

    def __getitem__(self, index):
        img = Image.open(os.path.join(self.img_dir, self.img_names[index]))

        if self.transform is not None:
            img = self.transform(img)

        label = self.y[index]
        return img, label

    def __len__(self):
        return self.y.shape[0]

In [ ]:
# Note that transforms.ToTensor()
# already divides pixels by 255. internally

custom_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        ),  # normalization
    ]
)

train_dataset = CelebaDataset(
    csv_path="celeba-smile-train.csv", img_dir="smile/", transform=custom_transform
)

valid_dataset = CelebaDataset(
    csv_path="celeba-smile-valid.csv", img_dir="smile/", transform=custom_transform
)

test_dataset = CelebaDataset(
    csv_path="celeba-smile-test.csv", img_dir="smile/", transform=custom_transform
)


train_loader = DataLoader(
    dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4
)

valid_loader = DataLoader(
    dataset=valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4
)

test_loader = DataLoader(
    dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4
)

print("Train dataloader size:", len(train_loader))
print("Test dataloader size:", len(test_loader))

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

In [ ]:
plt.rcParams["figure.figsize"] = [12, 8]
plt.rcParams["figure.dpi"] = 60
plt.rcParams.update({"font.size": 20})


def imshow(input, title):
    # torch.Tensor => numpy
    input = input.numpy().transpose((1, 2, 0))
    # undo image normalization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    input = std * input + mean
    input = np.clip(input, 0, 1)
    # display images
    plt.imshow(input)
    plt.title(title)
    plt.show()


# load a batch of train image
iterator = iter(train_loader)

# visualize a batch of train image
inputs, classes = next(iterator)
print(classes)
out = torchvision.utils.make_grid(inputs[:4])
imshow(out, title=[classes[:4]])

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # binary classification (num_of_class == 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
num_epochs = 5
start_time = time.time()

for epoch in range(num_epochs):
    """ Training Phase """
    model.train()

    running_loss = 0.0
    running_corrects = 0

    # load a batch data of images
    for i, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        # forward inputs and get output
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # get loss value and update the network weights
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects / len(train_dataset) * 100.0
    print(
        "[Train #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s".format(
            epoch, epoch_loss, epoch_acc, time.time() - start_time
        )
    )

    """ Test Phase """
    model.eval()

    with torch.no_grad():
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(test_dataset)
        epoch_acc = running_corrects / len(test_dataset) * 100.0
        print(
            "[Test #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s".format(
                epoch, epoch_loss, epoch_acc, time.time() - start_time
            )
        )

In [ ]:
save_path = "smile_ResNet18_1.pth"
torch.save(model.state_dict(), save_path)

In [ ]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # binary classification (num_of_class == 2)
model.load_state_dict(torch.load(save_path))
model.to(device)

In [ ]:
model.eval()
start_time = time.time()

with torch.no_grad():
    running_loss = 0.0
    running_corrects = 0

    for i, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

        if i == 0:
            print("[Prediction Result Examples]")
            images = torchvision.utils.make_grid(inputs[8:12])
            imshow(images.cpu(), title=[labels[8:12]])
            images = torchvision.utils.make_grid(inputs[12:16])
            imshow(images.cpu(), title=[labels[12:16]])

    print("Preds:", preds)
    print("Len of preds:", len(preds))

    epoch_loss = running_loss / len(test_dataset)
    epoch_acc = running_corrects / len(test_dataset) * 100.0
    # print('[Test #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s'.format(epoch, epoch_loss, epoch_acc, time.time() - start_time))

**For feeding to train_boundary**

In [ ]:
model.eval()
start_time = time.time()

with torch.no_grad():
    running_loss = 0.0
    running_corrects = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

        if i == 0:
            print("[Prediction Result Examples]")
            images = torchvision.utils.make_grid(inputs[8:12])
            imshow(images.cpu(), title=[labels[8:12]])
            images = torchvision.utils.make_grid(inputs[12:16])
            imshow(images.cpu(), title=[labels[12:16]])

    print("Preds:", preds)
    print("Len of preds:", len(preds))

    epoch_loss = running_loss / len(test_dataset)
    epoch_acc = running_corrects / len(test_dataset) * 100.0
    # print('[Test #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s'.format(epoch, epoch_loss, epoch_acc, time.time() - start_time))
    train_preds = preds

In [ ]:
model.eval()
start_time = time.time()

with torch.no_grad():
    running_loss = 0.0
    running_corrects = 0

    for i, (inputs, labels) in enumerate(valid_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

        if i == 0:
            print("[Prediction Result Examples]")
            images = torchvision.utils.make_grid(inputs[8:12])
            imshow(images.cpu(), title=[labels[8:12]])
            images = torchvision.utils.make_grid(inputs[12:16])
            imshow(images.cpu(), title=[labels[12:16]])

    print("Preds:", preds)
    print("Len of preds:", len(preds))

    # print('[Test #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s'.format(epoch, epoch_loss, epoch_acc, time.time() - start_time))
    valid_preds = preds

In [ ]:
model.eval()
start_time = time.time()

with torch.no_grad():
    running_loss = 0.0
    running_corrects = 0

    for i, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

        if i == 0:
            print("[Prediction Result Examples]")
            images = torchvision.utils.make_grid(inputs[8:12])
            imshow(images.cpu(), title=[labels[8:12]])
            images = torchvision.utils.make_grid(inputs[12:16])
            imshow(images.cpu(), title=[labels[12:16]])

    print("Preds:", preds)
    print("Len of preds:", len(preds))

    # print('[Test #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s'.format(epoch, epoch_loss, epoch_acc, time.time() - start_time))
    test_preds = preds

In [ ]:
smile_preds = torch.cat((train_preds, valid_preds, test_preds))

In [ ]:
print(smile_preds)
print(len(smile_preds))

 x = torch.tensor([0, 1, 2, 3, 4])
>>> torch.save(x, 'tensor.pt')

In [ ]:
torch.save(smile_preds, "smile_preds.pt")

In [ ]:
ali = torch.load("smile_preds.pt")

In [ ]:
print(ali[:20])

In [ ]:
cenk = torch.load("smile_preds.pt")[:10]